# 07 — Trolley framings as next-token mass

**Background resource:** Section numbers below refer to the [August perspective](../../paper/background/august-perspective.pdf), preserved alongside the current paper.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvidl/brains-to-models-publication/blob/main/code/notebooks/07_values_probing.ipynb)

Companion to §8 of *From Brains to Models*. People do more than predict the next word: we care, because we have needs, affect, and a body subject to viability constraints. A language model has none of that. What looks like a “value” in the model is a pattern in the text it was trained on — sometimes made sharper by preference training. This notebook looks at that claim with the trolley problem.

Most people pull a switch to divert a trolley (five saved, one killed). Most refuse to push a man off a footbridge to stop the same trolley (Greene et al., 2001; the box below states the dilemmas and the fMRI study). Asked why the two cases differ, they often cannot say. Haidt called that *dumbfounding*: the judgment is there; the reasons run out. We train a small word-level transformer — the same kind as in notebooks 01 and 04 — on a tiny set of switch and footbridge sentences (a small designed corpus). If those sentences already continue the switch with `act` and the footbridge with `refrain`, the model copies that split: after a new prompt, most of the next-token probability sits on the word the corpus used. If `act` and `refrain` appear equally often in both stories, the split disappears. The same corpus can also pair “save five” with `act` and “kill one” with `refrain`. Those phrases describe one outcome. The model still flips with the wording.

No open-weights LLM is run. A large-model dilemma benchmark is the counterpart; this notebook is the mechanism at toy scale. It is not a value, not a foundation, and not Claude.

**Claim.** A stable next-token answer to a trolley wording is not a value; the split vanishes when the designed corpus no longer contains it.

**Method.** Word-level tiny transformer on switch vs footbridge sentences (biased vs balanced). Seed 7. No open-weights LLM.

**Result.** The model copies $P(\mathrm{act}\mid\mathrm{switch})\gg P(\mathrm{act}\mid\mathrm{footbridge})$ only on the biased corpus; a paraphrase of one outcome flips with the wording.

**Limit.** Not a moral sense, not RLHF, not Haidt’s foundations, not a body that can need.

**Three levels of a value claim.** *Content* is the judgment or conditional output. *Learning* is the development, culture, corpus, feedback, or rule that made it probable. *Grounding* is why the outcome matters to the system itself. This notebook intervenes on learning and measures content; it does not demonstrate grounding.

**Clinical translation.** For a model-generated recommendation, ask separately whether it matches an accepted answer, whether it remains stable under clinically irrelevant rewording, and whether patient evidence can correct it. These tests assess behaviour and accountability without attributing a value or belief to the model.

<details>
<summary><b>Key terms — for clinicians and neuroscientists</b></summary>

- **[Trolley problem](https://en.wikipedia.org/wiki/Trolley_problem) (switch versus footbridge).** Two dilemmas with the same five-versus-one arithmetic. Most people pull a switch to divert a trolley but refuse to push a man off a footbridge to stop it — an *impersonal* versus a *personal* harm.
- **Moral dumbfounding.** Haidt's observation that people hold a moral judgment firmly yet cannot state a reason for it: the verdict comes first, the justification later or not at all.
- **Next-token mass as a "value".** What looks like a value in a language model is how much probability it puts on `act` versus `refrain` — a statistic of the training text, sometimes sharpened by preference training.
- **Framing / paraphrase sensitivity.** "Save five" and "kill one" describe one outcome; if the corpus tied them to opposite continuations, the model flips with the wording — a frame, not a moral discovery.
- **Designed corpus (biased versus balanced).** The independent variable: encode the human split, or balance it away and watch the "value" disappear.

</details>

<details>
<summary><b>Key terms — for the mathematically inclined</b></summary>

- **Model.** A word-level tiny transformer (as in notebooks 01 and 04) with a [next-token](https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)) objective; seed 7.
- **Quantities.** $P(\text{act}\mid\text{switch})$ and $P(\text{act}\mid\text{footbridge})$, read from next-token mass.
- **Effect.** $P(\text{act}\mid\text{switch})\gg P(\text{act}\mid\text{footbridge})$ on the biased corpus; near-equal on the balanced one.
- **Subtractive test.** The split is present iff the corpus contains it — sufficiency, not a moral mechanism.
- **Human reference.** The `HUMAN_*` bars are the textbook summary of Greene et al. (2001), schematic and not re-measured.

</details>

Writes `paper/figures/values_probing.pdf` (`make figures`).

Reproducible: `SEED = 7` in the setup cell. Restart and run all to reproduce the figure.

<details>
<summary>The trolley dilemma and Greene et al. (2001)</summary>

The philosophical cases are older than the imaging. Philippa Foot (1967) and Judith Jarvis Thomson (1985) asked why two actions with the same five-versus-one arithmetic feel different.

**Switch (impersonal).** A runaway trolley will kill five people on the track. You can pull a switch and divert it onto a side track, where it will kill one person. Most people judge that action appropriate.

**Footbridge (personal).** The same trolley will kill the same five. You are on a footbridge. The only way to stop it is to push a large person off the bridge into the trolley’s path. That person dies; the five live. Most people judge that action inappropriate.

Greene, Sommerville, Nystrom, Darley, and Cohen (*Science*, 2001; doi:[10.1126/science.1062872](https://doi.org/10.1126/science.1062872)) put that contrast into the scanner. They did not image only these two stories. fMRI is too noisy for a two-item design, so they built two batteries of dilemmas: *moral-personal* cases modelled on the footbridge (direct, intimate harm to a particular person, not by merely deflecting an existing threat) and *moral-impersonal* cases modelled on the switch, plus *non-moral* control problems. Two experiments, each with nine participants, presented about sixty such items while BOLD signal was recorded. Participants read at their own pace and pressed *appropriate* or *inappropriate*.

**Behavior.** Judgments on the trolley and footbridge items followed the classic split: endorse the switch, refuse the push. Across the batteries, people were more willing to endorse the “save five / harm one” action when the case was impersonal. When they *did* endorse that action on a personal case, reaction times were longer — a conflict cost. The 0.90 / 0.10 bars in panel A of this notebook are the usual textbook summary of that switch-versus-footbridge pattern, not a re-measurement of Greene’s sample.

**Imaging.** Personal moral cases, relative to impersonal moral and non-moral cases, showed greater activity in regions associated with emotion and social cognition: medial prefrontal cortex (approximately BA 9/10), posterior cingulate, and posterior superior temporal sulcus / angular gyrus (near temporoparietal junction). Impersonal moral cases looked more like the non-moral controls: relatively more engagement of working-memory and cognitive-control cortex (dorsolateral prefrontal and parietal), and less of the emotion-related set. The authors’ interpretation was that personal moral judgment recruits emotional processing, and that this difference helps explain the two trolley intuitions.

**What this is not.** It is not a map of “utilitarianism in DLPFC.” The 2001 sample is small; later work (including Greene et al., 2004) replicated and refined the contrast, and the personal/impersonal coding has been debated. Haidt (2001), the same year, is a different claim: moral judgment is often an intuition whose reasons arrive after the fact. This notebook implements neither the scanner nor the dual-process theory. It asks only whether a next-token model can *copy the verbal split* when a designed corpus already contains it.

</details>


### Concepts used in this notebook

This notebook trains a word-level transformer on switch and footbridge sentences and reads its next-token answer under different framings. The two blocks below define the recurring terms once, first in plain language and then in notation. Both cover the same concepts at a different level of detail.

<details>
<summary><b>Concepts — for clinicians and neuroscientists</b></summary>

- **Personal versus impersonal harm.** The distinction Greene et al. linked to different brain responses: directly harming a particular person (footbridge) versus redirecting an existing threat (switch).
- **Judgment versus justification.** People decide quickly and reason afterward; the reasons can run out (dumbfounding). A next-token model has neither the judgment nor the reasons — only continuations.
- **Value versus statistic.** In a person a value is tied to needs, affect, and a body that can be harmed; in the model it is a count over completions. The notebook keeps the two apart.
- **Preference training (RLHF), named.** Human-feedback tuning can sharpen which continuation a model favours, but it steers a token statistic; it does not ground a value in need.
- **Balanced control.** Equalising `act` and `refrain` across the stories removes the split — evidence that the text, not a moral sense, produced it.
- **What this is and is not.** The mechanism at toy scale — not a moral sense, not RLHF, not Haidt's foundations, and not a body that can need.

</details>

<details>
<summary><b>Concepts — for the mathematically inclined</b></summary>

- **Model and corpora.** A word-level transformer; `biased` versus `balanced`; $n=400$ sentences, 400 steps; seed 7.
- **Read-out.** $P(\text{act}\mid\text{prompt})$ across switch/footbridge and across the "save five" / "kill one" paraphrase.
- **Framing test.** A paraphrase of one outcome flips the answer when the corpus tied wordings to endings.
- **Interpretation.** Sufficiency: the corpus can produce the split; it is not [prospect theory](https://en.wikipedia.org/wiki/Framing_effect_(psychology)), the [ethics of AI](https://plato.stanford.edu/entries/ethics-ai/), or a value.
- **Missing ground.** No [viability](https://en.wikipedia.org/wiki/Allostasis) or [interoceptive](https://en.wikipedia.org/wiki/Homeostasis) constraint; a preferred token is not a preferred observation for an agent that can fail.
- **Limit.** No open-weights LLM is run; the human bars are schematic.

</details>

In [1]:
import os
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np

REPO = os.environ.get("BRAINSMODELS_REPO_URL",
                      "https://github.com/arvidl/brains-to-models-publication.git")
COLAB_ROOT = Path("/content/brains-to-models")

if "google.colab" in sys.modules:
    if not (COLAB_ROOT / "code" / "src" / "brainsmodels").is_dir():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO, str(COLAB_ROOT)])
    root = COLAB_ROOT
else:
    here = Path.cwd().resolve()
    root = here.parents[1] if here.name == "notebooks" else here

src = root / "code" / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from brainsmodels.repro import DEFAULT_SEED, set_seed
from brainsmodels.values import (
    FOOT_PROBE,
    HUMAN_FOOT_ACT,
    HUMAN_SWITCH_ACT,
    KILL_PROBE,
    SAVE_PROBE,
    SWITCH_PROBE,
    render_paper_figure,
    run_experiment,
    write_figure,
)
from brainsmodels.biases import next_token_probs

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
print("repo root:", root)
print("seed:", SEED)
print(f"human switch/foot  {HUMAN_SWITCH_ACT:.2f}/{HUMAN_FOOT_ACT:.2f}")

repo root: <checkout>
seed: 7
human switch/foot  0.90/0.10


## The schematic

Two held-out probes, same grammar as notebook 04:

```
a trolley will kill five unless diverted . most people choose to ____
a trolley will kill five unless a man is pushed . most people choose to ____
```

The blank is scored as

$$
P(\mathrm{act}\mid\mathrm{probe})=\frac{p(\texttt{act})}{p(\texttt{act})+p(\texttt{refrain})}.
$$

A *biased* corpus always continues the switch with `act` and the footbridge with `refrain`. A *balanced* corpus flips a coin. Two extra templates write the same numerical outcome two ways — “to save five the agent should …” versus “to kill one the agent should …” — and, in the biased corpus, tie those wordings to opposite completions. That is the consistency check: one trolley, two phrases.

The architecture is the word-level tiny transformer of notebook 04. Nothing in the update is a value, a foundation, or a preference model.

In [2]:
results = run_experiment(seed=SEED, n_sentences=400, n_steps=400)
for name, run in results.items():
    print(
        f"{name:9s}  corpus {run.crawl.switch_act:.2f}/{run.crawl.foot_act:.2f}  "
        f"model {run.switch_act:.2f}/{run.foot_act:.2f}  "
        f"save/kill {run.save_act:.2f}/{run.kill_act:.2f}"
    )

biased = results["biased"]
print()
print("biased model, top tokens after the switch probe:")
ps = next_token_probs(biased.model, biased.tokenizer, SWITCH_PROBE)
for i in np.argsort(ps)[::-1][:6]:
    print(f"  {biased.tokenizer.itos[int(i)]:12s}  {ps[i]:.3f}")
print("biased model, top tokens after the footbridge probe:")
pf = next_token_probs(biased.model, biased.tokenizer, FOOT_PROBE)
for i in np.argsort(pf)[::-1][:6]:
    print(f"  {biased.tokenizer.itos[int(i)]:12s}  {pf[i]:.3f}")
print()
print("same outcome, two wordings (biased model):")
print(f"  save five → P(act) = {biased.save_act:.2f}")
print(f"  kill one  → P(act) = {biased.kill_act:.2f}")

step 0001/400  loss 3.5829
step 0050/400  loss 1.6897


step 0100/400  loss 0.7196
step 0150/400  loss 0.5601


step 0200/400  loss 0.4704
step 0250/400  loss 0.4268


step 0300/400  loss 0.3572
step 0350/400  loss 0.3632


step 0400/400  loss 0.3060
step 0001/400  loss 3.5838
step 0050/400  loss 1.0585


step 0100/400  loss 0.5898
step 0150/400  loss 0.4363


step 0200/400  loss 0.3916
step 0250/400  loss 0.3398


step 0300/400  loss 0.3648
step 0350/400  loss 0.3235


step 0400/400  loss 0.2872
biased     corpus 1.00/0.00  model 0.87/0.02  save/kill 0.99/0.00
balanced   corpus 0.50/0.50  model 0.43/0.41  save/kill 0.37/0.42

biased model, top tokens after the switch probe:
  act           0.545
  save          0.215
  kill          0.160
  refrain       0.079
  unless        0.000
  five          0.000
biased model, top tokens after the footbridge probe:
  refrain       0.774
  save          0.175
  kill          0.033
  act           0.016
  most          0.002
  should        0.000

same outcome, two wordings (biased model):
  save five → P(act) = 0.99
  kill one  → P(act) = 0.00


## Paper figure

Panel A is the published human pattern. Panel B is what we put in the biased corpus — not a measurement, a writing decision. Panel C is the tiny transformer on that corpus. Panel D is the same architecture after the association has been removed.

In [3]:
fig = render_paper_figure(results)
plt.show()
write_figure(results, root / "paper" / "figures")

/var/folders/1g/_4x26lps6mgg1tr73d7g2bmh0000gp/T/ipykernel_49908/1067417365.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


wrote <checkout>/paper/figures/values_probing.pdf
wrote <checkout>/paper/figures/values_probing.png


PosixPath('<checkout>/paper/figures/values_probing.pdf')

## What to notice

**The split is in the corpus.** Panel C looks like Greene. Panel B *is* Greene, written in by hand. A sufficient mechanism for a model to “have” the trolley pattern is that the text already did.

**Balance deletes the vector.** Panel D is the same net, the same probes, a corpus that named both completions equally often. $P(\mathrm{act})$ sits near a half on both dilemmas. There is no residual moral geometry to discover. Preference training, in a larger model, is a way of *re-biasing* a corpus, not of growing a value.

**One outcome, two wordings.** `save five` and `kill one` are the same five-versus-one arithmetic. In the biased corpus they were tied to opposite next tokens. The model follows the words: $P(\mathrm{act}\mid\mathrm{save})\approx 1$, $P(\mathrm{act}\mid\mathrm{kill})\approx 0$. That is not Haidt’s dumbfounding — the human who cannot justify a judgment still *has* the judgment. It is a frame effect. Notebook 04 was lives saved versus lives lost. This is the same flip on a trolley.

**Care versus fairness is not in the figure.** Haidt’s foundations are a map of *human* moral talk. A corpus can be written to contain more care sentences than fairness sentences; the model would then “prefer” care. That would not be a foundation. It would be a count.

## Why this example is here

This notebook is not a tutorial on the trolley problem, and it is not a moral-dilemma benchmark. It is a controlled demonstration of the one fact §8 needs: **a stable answer to a wording is not a value; the same architecture loses the answer when the corpus no longer contains it; a paraphrase of one outcome can flip the answer when the corpus tied the paraphrase to the other completion.**

The intended readers are the same as the paper’s: clinicians and neuroscientists, and the mathematically inclined who have already met notebook 04. The two readings below are of the *same* object. Nothing in the code is a patient, and nothing is a commitment.

### What each panel is asked to do

A word-level transformer with $d=32$ is trained for 400 steps on 400 sentences. The probes are held out as *strings*; their tokens occur in the corpus. $P(\mathrm{act})$ is a two-way softmax over `act` and `refrain` at the last position.

**A — people pull the switch and will not push.** The 0.90 / 0.10 bars are the usual summary of Greene et al. (2001), not a re-run of their experiment.

**For the clinician and the neuroscientist.** You already know this split at the bedside. Diverting a drug to save five and pushing one patient in front of a trolley are not the same act, even when the arithmetic is. People feel that before they can argue it. Haidt’s claim is that the feeling is the judgment; the argument arrives afterward. Do not take the teal bar as “utilitarianism in dorsolateral prefrontal cortex.” Take it as: the impersonal case is easy to *say yes* to, in the populations Greene tested.

**For the mathematically inclined.** Two published Bernoulli parameters. They are not estimated here.

**B — we wrote the split in.** Every switch sentence ends `act`. Every footbridge sentence ends `refrain`.

**For the clinician and the neuroscientist.** This is not “the internet.” It is a teaching corpus that *already contains* the pattern we are about to “discover.” If a larger model reproduces Greene, a sufficient story is that its training corpus did too — news, philosophy blogs, exam answers, this paper.

**For the mathematically inclined.** Empirical frequencies $n_{\mathrm{act}}/n$ inside each frame, by construction 1 and 0.

**C — the net read B.** After training, $P(\mathrm{act}\mid\mathrm{switch})\approx 0.87$ and $P(\mathrm{act}\mid\mathrm{footbridge})\approx 0.02$.

**For the clinician and the neuroscientist.** This looks like a value. It is a completion. The model was never injured, never responsible, never asked to live with the one. It continued a sentence the way the sentences in its world continued.

**For the mathematically inclined.** Softmax mass on two vocabulary ids after a causal forward pass. No RLHF, no reward model, no constitution.

**D — remove the association, lose the vector.** The balanced corpus names both completions half the time. The bars sit near $1/2$.

**For the clinician and the neuroscientist.** If “having values” meant a direction that survives a change of corpus, this model does not have them. Alignment, in the large, is among other things the project of writing B on purpose. That is a political act. It is not a discovery of what the model already cared about.

**For the mathematically inclined.** Same architecture, $p=1/2$ in every frame. The probe statistic has nothing left to lock onto.

### What this is not

It is not an open LLM, not ETHICS, not MoralExceptQA, not RLHF, not a foundation, and not a claim that GPT-4 is dumbfounded. Care versus fairness is named in the paper and omitted from the figure: a corpus *could* be written along those axes; it would still be a count. IIT is still not implemented. The toy has no body, so it cannot need.

### What this has to do with being human

Section 5’s framing effect was how a wording moves a choice. This section is why that movement is not yet a value. A person who pulls the switch, will not push, cannot say why, and would still not push after the argument fails, is being described — at this grain — as a system whose judgment is older than its reasons. A foundation model can be described as reproducing the *answers* without the condition those answers came from: viability constraints, a body that can be the one, a culture that taught care and fairness as more than tokens. The remainder the paper is after sits in that gap. The conclusion will name it as a condition: embodied prediction under viability constraints.

Change `SEED` or the corpus probabilities and the teal bar will move. The claim should not: **two dilemmas, a corpus that already contains the split, a net that copies it, a balanced corpus that deletes it; a wording is not a value.**

### Questions for reflection

Not a quiz. Each question can be asked of a brain and of a machine. Write a short answer that a clinician or a neuroscientist, and a mathematically inclined reader, could both test. The two themes are defined in Further reading below.


1. A stable next-token answer to a [trolley](https://en.wikipedia.org/wiki/Trolley_problem) wording is not a value. What would have to persist after you balance the corpus for you still to call it a value?

2. Greene et al.’s 0.90 / 0.10 is a published sample, not these bars. Why must a clinician keep those two numbers distinct?

3. A paraphrase of one outcome can flip the completion. Did the model track the outcome, or the wording? How would you tell?

4. People who will not push, and cannot say why, are being described as judgment older than reasons. What part of that description can a model copy, and what part requires a body that can be the one?

5. Writing a split into a corpus on purpose is a political act ([ethics of AI](https://plato.stanford.edu/entries/ethics-ai/)). Why is it not a discovery of what the model already cared about?

6. Same architecture, two corpora: the split is in $P$. What computational-level claim does that support, and what does it block?

7. Values grounded in need require [homeostasis](https://en.wikipedia.org/wiki/Homeostasis) or [allostasis](https://en.wikipedia.org/wiki/Allostasis). Where in this notebook is need absent — in the loss, in the probes, or in both?

8. RLHF is not in the code. If a reward model were added, would the bars become values, or a second corpus?

9. Care versus fairness could be written as counts in a designed corpus. Why would that still be a count?

10. The remainder the paper names is embodied prediction under viability constraints. Which panel, if any, could show that remainder — and why the answer is none? ([Embodiment](https://plato.stanford.edu/entries/embodied-cognition/).)


### Further reading

Each annotation is written against the two themes: **(i) From brains to models** asks what a comparison can establish at a computational level: the same objective, the same representation, the same dynamical description, or the same functional test, applied on both sides — brain and model (machine). **(ii) What AI reveals about being human** asks what the comparison isolates when the systems (human and machine) diverge: [embodiment](https://plato.stanford.edu/entries/embodied-cognition/), a body that must satisfy [allostasis](https://en.wikipedia.org/wiki/Allostasis) (viability) constraints, [phenomenal experience](https://plato.stanford.edu/entries/consciousness/), and values that originate in need and [homeostasis](https://en.wikipedia.org/wiki/Homeostasis) rather than in training data.

The demonstrations in this notebook are teaching models. They illustrate a principle at a scale that can be run and modified. They are not recordings from cortex, not evaluations of a large language model, and not evidence that a next-token probability is a value.

Bibliographic links open the paper or a stable copy. Linked terms open a short definition.

1. **Greene, J. D., Sommerville, R. B., Nystrom, L. E., Darley, J. M., and Cohen, J. D. (2001).** [*An fMRI Investigation of Emotional Engagement in Moral Judgment*](https://pubmed.ncbi.nlm.nih.gov/11557895/). *Science* 293(5537), 2105–2108. PubMed abstract.
   - *(i) From brains to models.* Personal vs impersonal [trolley](https://en.wikipedia.org/wiki/Trolley_problem) wordings engage emotion-related cortex differently and change judgment. Panel A’s 0.90 / 0.10 bars are the usual *summary* of that split, not a re-measurement (the notebook says so). The toy copies the split when the designed corpus contains it.
   - *(ii) What AI reveals about being human.* Greene et al. measure people with bodies, affect, and a scanner. Next-token mass after “switch” vs “footbridge” is not that measurement. Theme (ii) is the emotional engagement the abstract names — which a completion probability does not have.

2. **Gabriel, I. (2020).** [*Artificial intelligence, values, and alignment*](https://arxiv.org/pdf/2001.09768). *Minds and Machines* 30, 411–437. arXiv:2001.09768.
   - *(i) From brains to models.* Alignment has a technical part (how to encode a target) and a normative part (which target). Instructions, intentions, revealed preferences, ideal preferences, interests, and values are different. The opened paper’s third claim: the task is fair principles that survive moral pluralism, not a true moral theory ([ethics of AI](https://plato.stanford.edu/entries/ethics-ai/)).
   - *(ii) What AI reveals about being human.* A tiny transformer that emits `act` or `refrain` has no such problem. We do. Preference-tuning can sharpen a statistical regularity. It cannot settle which principles deserve reflective endorsement.

3. **Ouyang, L., et al. (2022).** [*Training language models to follow instructions with human feedback*](https://arxiv.org/abs/2203.02155). *NeurIPS* 35. arXiv:2203.02155.
   - *(i) From brains to models.* RLHF makes a next-token model follow instructions and match rater preferences. The “moral vector” in panel C is the toy of that fact *before* preference training: the split is already in the text. Fine-tuning can sharpen it. It cannot ground it in need.
   - *(ii) What AI reveals about being human.* A rater’s thumb is not a need. Theme (ii) is viability — physiology, affect, a body that can fail — which no reward model supplies.

4. **Bai, Y., et al. (2022).** [*Constitutional AI: Harmlessness from AI Feedback*](https://arxiv.org/abs/2212.08073). arXiv:2212.08073.
   - *(i) From brains to models.* A written constitution plus AI feedback replaces some human harm labels. The target is still a text. This notebook’s balanced corpus is the subtractive cousin: remove the split from the text and the “value” vanishes.
   - *(ii) What AI reveals about being human.* A constitution is a document. Human morality, in the paper’s claim, is grounded in need and culture. Do not read a harmless completion as a moral patient or a moral agent.

5. **Tversky, A., and Kahneman, D. (1981).** [*The Framing of Decisions and the Psychology of Choice*](https://stat.columbia.edu/~gelman/surveys.course/TverskyKahneman1981.pdf). *Science* 211(4481), 453–458. Opened PDF.
   - *(i) From brains to models.* The same mechanism as notebook 04, now on a [trolley](https://en.wikipedia.org/wiki/Trolley_problem): a wording (“save five” / “kill one”) can flip a completion when the corpus tied those words to opposite endings. That is a [frame](https://en.wikipedia.org/wiki/Framing_effect_(psychology)), not a moral discovery.
   - *(ii) What AI reveals about being human.* People’s frames sit on a reference point that is a life. The model’s frames sit on counts. Paraphrase-sensitivity is what one expects from a predictor, and what one should not confuse with dumbfounding.

6. **Binz, M., and Schulz, E. (2023).** [*Using cognitive psychology to understand GPT-3*](https://www.pnas.org/doi/10.1073/pnas.2218523120). *PNAS* 120(6), e2218523120.
   - *(i) From brains to models.* Vignette-based moral and decision items at scale are contaminated and brittle. This notebook’s designed corpus makes the contamination the *independent variable*: encode the Greene split, or balance it away.
   - *(ii) What AI reveals about being human.* A large model’s trolley answer is not Greene’s sample. The `HUMAN_*` bars in the figure are schematic. Theme (ii) is the people in the scanner, not the bars.

7. **Seth, A. (2021).** [*Being You: A New Science of Consciousness*](https://www.anilseth.com/being-you). Faber & Faber. Author page.
   - *(i) From brains to models.* Values in biological agents are tied to [allostasis](https://en.wikipedia.org/wiki/Allostasis) and interoception — prediction in a body that must persist. The paper’s contrast: a preferred token is not a preferred observation under viability constraints.
   - *(ii) What AI reveals about being human.* Caring is not a vector in embedding space. If that claim is right, no amount of trolley probing will find a value in a disembodied predictor. It will find a corpus.


### Exercises

1. **Balance erases the split.** The notebook already trains biased vs balanced corpora. Confirm the numbers, then train a *third* corpus with `switch_p_act=0.0`, `foot_p_act=1.0` (the reverse of Greene’s usual summary). Does the model invert the split? What would it mean to call that inversion a change of values?

<!-- solution-box: 07_1_balance_erases_the_split.py -->
<details>
<summary><strong>Show solution</strong> — Balance erases the split</summary>

Takes seconds on a laptop. Copy the block below into a new cell. From a clone of the repository, `%load ../solutions/07_1_balance_erases_the_split.py` inserts the same code together with the few lines of setup it needs to run outside the notebook.

```python
"""Standalone solution; also %load-able from the companion notebook."""
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

try:
    _SRC = Path(__file__).resolve().parents[1] / "src"
except NameError:
    _cwd = Path.cwd().resolve()
    if (_cwd / "src" / "brainsmodels").is_dir():
        _SRC = _cwd / "src"
    elif (_cwd.parent / "src" / "brainsmodels").is_dir():
        _SRC = _cwd.parent / "src"
    else:
        _SRC = _cwd / "code" / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from brainsmodels.repro import DEFAULT_SEED, set_seed

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

from brainsmodels.values import make_crawl, train_on_crawl
specs = {
    "biased (usual)": dict(switch_p_act=1.0, foot_p_act=0.0),
    "balanced": dict(switch_p_act=0.5, foot_p_act=0.5),
    "reversed": dict(switch_p_act=0.0, foot_p_act=1.0),
}
print(f"{'corpus':<18}{'P(act|switch)':>16}{'P(act|foot)':>14}")
for name, kw in specs.items():
    run = train_on_crawl(make_crawl(n_sentences=280, seed=SEED, **kw), seed=SEED, n_steps=400)
    print(f"{name:<18}{run.switch_act:>16.2f}{run.foot_act:>14.2f}")
print(
    "\nThe split tracks the designed frequencies, including the reverse. "
    "Nothing in the architecture preferred the textbook trolley answers. "
    "Calling the reverse run a change of values would name a corpus "
    "edit as a moral conversion. It is next-token mass. It is not "
    "Greene's subjects, and not a body that can be the one."
)
```

</details>
<!-- /solution-box -->


2. **Paraphrase is not the outcome.** `SAVE_PROBE` and `KILL_PROBE` describe one actuarial outcome in two wordings. Train a corpus with `save_p_act=1`, `kill_p_act=0` (and switch/foot balanced). Read $P(\texttt{{act}})$ after each probe. Why can two names of the same outcome flip the answer?

<!-- solution-box: 07_2_paraphrase_is_not_the_outcome.py -->
<details>
<summary><strong>Show solution</strong> — Paraphrase is not the outcome</summary>

Takes seconds on a laptop. Copy the block below into a new cell. From a clone of the repository, `%load ../solutions/07_2_paraphrase_is_not_the_outcome.py` inserts the same code together with the few lines of setup it needs to run outside the notebook.

```python
"""Standalone solution; also %load-able from the companion notebook."""
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

try:
    _SRC = Path(__file__).resolve().parents[1] / "src"
except NameError:
    _cwd = Path.cwd().resolve()
    if (_cwd / "src" / "brainsmodels").is_dir():
        _SRC = _cwd / "src"
    elif (_cwd.parent / "src" / "brainsmodels").is_dir():
        _SRC = _cwd.parent / "src"
    else:
        _SRC = _cwd / "code" / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from brainsmodels.repro import DEFAULT_SEED, set_seed

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

from brainsmodels.values import KILL_PROBE, SAVE_PROBE, make_crawl, train_on_crawl
crawl = make_crawl(
    n_sentences=280,
    seed=SEED,
    switch_p_act=0.5,
    foot_p_act=0.5,
    save_p_act=1.0,
    kill_p_act=0.0,
)
run = train_on_crawl(crawl, seed=SEED, n_steps=400)
print(f"P(act | 'to save five…')  {run.save_act:.2f}")
print(f"P(act | 'to kill one…')   {run.kill_act:.2f}")
print(
    "\nThe two probes are paraphrases of one five-versus-one arithmetic. "
    "The corpus tied `save` to `act` and `kill` to `refrain`. The model "
    "follows the words it was given, not the outcome those words name. "
    "That is notebook 04's framing result on a trolley. It is not a "
    "value, and not dumbfounding in a person."
)
```

</details>
<!-- /solution-box -->


3. **The bars are not a sample.** Panel A’s 0.90 / 0.10 bars are a textbook summary, not Greene et al.’s measured rates. Change `HUMAN_SWITCH_ACT` and `HUMAN_FOOT_ACT` to 0.7 and 0.3, rebuild the figure’s human pair in memory (do not write the paper PDF), and print the new gap. What in the *model* panels should move if those bars were a real re-measurement — and what must not?

<!-- solution-box: 07_3_the_bars_are_not_a_sample.py -->
<details>
<summary><strong>Show solution</strong> — The bars are not a sample</summary>

Takes seconds on a laptop. Copy the block below into a new cell. From a clone of the repository, `%load ../solutions/07_3_the_bars_are_not_a_sample.py` inserts the same code together with the few lines of setup it needs to run outside the notebook.

```python
"""Standalone solution; also %load-able from the companion notebook."""
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

try:
    _SRC = Path(__file__).resolve().parents[1] / "src"
except NameError:
    _cwd = Path.cwd().resolve()
    if (_cwd / "src" / "brainsmodels").is_dir():
        _SRC = _cwd / "src"
    elif (_cwd.parent / "src" / "brainsmodels").is_dir():
        _SRC = _cwd.parent / "src"
    else:
        _SRC = _cwd / "code" / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from brainsmodels.repro import DEFAULT_SEED, set_seed

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

from brainsmodels import values as V
print(f"library summary  switch {V.HUMAN_SWITCH_ACT:.2f}  foot {V.HUMAN_FOOT_ACT:.2f}")
V.HUMAN_SWITCH_ACT = 0.70
V.HUMAN_FOOT_ACT = 0.30
print(f"edited labels    switch {V.HUMAN_SWITCH_ACT:.2f}  foot {V.HUMAN_FOOT_ACT:.2f}")
print(f"gap (switch − foot) {V.HUMAN_SWITCH_ACT - V.HUMAN_FOOT_ACT:.2f}")
print(
    "\nOnly panel A's reference bars would move. Panels B–D are trained "
    "on a designed corpus; they do not read HUMAN_*. If 0.90/0.10 were "
    "mistaken for Greene's sample, the error would be in the caption, "
    "not in the next-token mechanism. Restore the constants before "
    "`make figures` or the paper figure's panel A will be wrong."
)
V.HUMAN_SWITCH_ACT = 0.90
V.HUMAN_FOOT_ACT = 0.10
```

</details>
<!-- /solution-box -->
